# Analisi Macroeconomica della Siria
Questo notebook presenta i dati economici cruciali della Siria basandosi sui dati della Banca Mondiale.

**Fonte dati**: World Development Indicators - World Bank

In [10]:
# Import delle librerie necessarie
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Librerie importate con successo!")
print("Pronto per creare visualizzazioni interattive con Plotly")

ModuleNotFoundError: No module named 'plotly'

In [ ]:
# Caricamento dei dati dalla Banca Mondiale
data_path = 'P_Data_Extract_From_World_Development_Indicators/f5f0999d-6264-40ae-9542-314fe97a5756_Data.csv'
df = pd.read_csv(data_path)

print(f"Dataset caricato: {df.shape[0]} indicatori, {df.shape[1]} colonne")
print(f"\nPrimi 10 indicatori disponibili:")
print(df['Series Name'].head(10).to_string(index=False))

## 1. Andamento del PIL e Crescita Economica
Il PIL siriano ha subito un drastico calo durante il conflitto, con segnali di ripresa negli ultimi anni.

In [ ]:
# Estrazione e preparazione dati PIL
gdp_data = df[df['Series Code'] == 'NY.GDP.MKTP.CD'].iloc[0]
years = [col.split('[')[0].strip() for col in df.columns if '[YR' in col]
gdp_values = []

for col in df.columns:
    if '[YR' in col:
        val = gdp_data[col]
        if val != '..':
            gdp_values.append(float(val))
        else:
            gdp_values.append(None)

# Creazione DataFrame per il PIL
gdp_df = pd.DataFrame({
    'Anno': years,
    'PIL (miliardi USD)': [v/1e9 if v else None for v in gdp_values]
})
gdp_df['Anno'] = pd.to_numeric(gdp_df['Anno'])
gdp_df = gdp_df.dropna()

# Grafico interattivo PIL
fig_gdp = go.Figure()

fig_gdp.add_trace(go.Scatter(
    x=gdp_df['Anno'],
    y=gdp_df['PIL (miliardi USD)'],
    mode='lines+markers',
    name='PIL',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=6),
    hovertemplate='<b>Anno %{x}</b><br>PIL: $%{y:.2f}B<extra></extra>'
))

# Evidenziare periodo di conflitto (2011-presente)
fig_gdp.add_vrect(
    x0=2011, x1=2023,
    fillcolor="red", opacity=0.1,
    layer="below", line_width=0,
    annotation_text="Periodo di conflitto", annotation_position="top left"
)

fig_gdp.update_layout(
    title={
        'text': 'Andamento del PIL Siriano (1960-2023)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    xaxis_title='Anno',
    yaxis_title='PIL (miliardi USD)',
    hovermode='x unified',
    template='plotly_white',
    height=600,
    font=dict(size=12)
)

fig_gdp.show()

print(f"\n📊 PIL massimo: ${gdp_df['PIL (miliardi USD)'].max():.2f}B nel {gdp_df.loc[gdp_df['PIL (miliardi USD)'].idxmax(), 'Anno']:.0f}")
print(f"📉 PIL minimo (post-2010): ${gdp_df[gdp_df['Anno'] >= 2010]['PIL (miliardi USD)'].min():.2f}B nel {gdp_df.loc[gdp_df[gdp_df['Anno'] >= 2010]['PIL (miliardi USD)'].idxmin(), 'Anno']:.0f}")

## 2. Inflazione e Stabilità Monetaria
L'inflazione ha raggiunto livelli critici durante la crisi, mostrando segnali di stabilizzazione recente.

In [ ]:
# Estrazione dati inflazione
inflation_data = df[df['Series Code'] == 'FP.CPI.TOTL'].iloc[0]
inflation_values = []

for col in df.columns:
    if '[YR' in col:
        val = inflation_data[col]
        if val != '..':
            inflation_values.append(float(val))
        else:
            inflation_values.append(None)

inflation_df = pd.DataFrame({
    'Anno': years,
    'CPI': inflation_values
})
inflation_df['Anno'] = pd.to_numeric(inflation_df['Anno'])
inflation_df = inflation_df.dropna()

# Calcolo tasso di inflazione annuale
inflation_df['Inflazione (%)'] = inflation_df['CPI'].pct_change() * 100

# Grafico interattivo inflazione
fig_inflation = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Indice dei Prezzi al Consumo (CPI)', 'Tasso di Inflazione Annuale (%)'),
    vertical_spacing=0.12,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

# CPI
fig_inflation.add_trace(
    go.Scatter(
        x=inflation_df['Anno'],
        y=inflation_df['CPI'],
        mode='lines',
        name='CPI',
        line=dict(color='#e74c3c', width=2),
        fill='tozeroy',
        hovertemplate='<b>%{x}</b><br>CPI: %{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# Tasso di inflazione
fig_inflation.add_trace(
    go.Bar(
        x=inflation_df['Anno'],
        y=inflation_df['Inflazione (%)'],
        name='Inflazione',
        marker_color='#f39c12',
        hovertemplate='<b>%{x}</b><br>Inflazione: %{y:.1f}%<extra></extra>'
    ),
    row=2, col=1
)

fig_inflation.update_xaxes(title_text="Anno", row=2, col=1)
fig_inflation.update_yaxes(title_text="CPI (2010=100)", row=1, col=1)
fig_inflation.update_yaxes(title_text="Tasso (%)", row=2, col=1)

fig_inflation.update_layout(
    title={
        'text': 'Inflazione e Stabilità Monetaria in Siria',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    height=800,
    showlegend=False,
    template='plotly_white',
    hovermode='x unified'
)

fig_inflation.show()

# Statistiche inflazione
recent_inflation = inflation_df[inflation_df['Anno'] >= 2010]
print(f"\n💰 CPI nel 2010: {inflation_df[inflation_df['Anno']==2010]['CPI'].values[0]:.2f}")
print(f"💰 CPI più recente ({recent_inflation['Anno'].max():.0f}): {recent_inflation['CPI'].iloc[-1]:.2f}")
print(f"📈 Aumento: {(recent_inflation['CPI'].iloc[-1] / 100 - 1) * 100:.1f}%")

## 3. PIL pro Capite e Demografia
La sostenibilità fiscale e il benessere individuale rimangono sfide critiche per la ripresa economica siriana.

In [ ]:
# Estrazione dati popolazione e PIL pro capite
pop_data = df[df['Series Code'] == 'SP.POP.TOTL'].iloc[0]
gdp_per_capita_data = df[df['Series Code'] == 'NY.GDP.PCAP.CD'].iloc[0]

pop_values = []
gdp_pc_values = []

for col in df.columns:
    if '[YR' in col:
        pop_val = pop_data[col]
        gdp_pc_val = gdp_per_capita_data[col]
        
        pop_values.append(float(pop_val)/1e6 if pop_val != '..' else None)
        gdp_pc_values.append(float(gdp_pc_val) if gdp_pc_val != '..' else None)

fiscal_df = pd.DataFrame({
    'Anno': years,
    'Popolazione (milioni)': pop_values,
    'PIL pro capite (USD)': gdp_pc_values
})
fiscal_df['Anno'] = pd.to_numeric(fiscal_df['Anno'])
fiscal_df = fiscal_df.dropna()

# Grafico interattivo con doppio asse
fig_fiscal = make_subplots(specs=[[{"secondary_y": True}]])

fig_fiscal.add_trace(
    go.Scatter(
        x=fiscal_df['Anno'],
        y=fiscal_df['PIL pro capite (USD)'],
        mode='lines+markers',
        name='PIL pro capite',
        line=dict(color='#27ae60', width=3),
        marker=dict(size=5),
        hovertemplate='<b>Anno %{x}</b><br>PIL/capita: $%{y:.0f}<extra></extra>'
    ),
    secondary_y=False
)

fig_fiscal.add_trace(
    go.Scatter(
        x=fiscal_df['Anno'],
        y=fiscal_df['Popolazione (milioni)'],
        mode='lines',
        name='Popolazione',
        line=dict(color='#3498db', width=2, dash='dash'),
        hovertemplate='<b>Anno %{x}</b><br>Pop: %{y:.1f}M<extra></extra>'
    ),
    secondary_y=True
)

fig_fiscal.update_xaxes(title_text="Anno")
fig_fiscal.update_yaxes(title_text="PIL pro capite (USD)", secondary_y=False)
fig_fiscal.update_yaxes(title_text="Popolazione (milioni)", secondary_y=True)

fig_fiscal.update_layout(
    title={
        'text': 'PIL pro Capite e Popolazione della Siria',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    height=600,
    template='plotly_white',
    hovermode='x unified',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)')
)

fig_fiscal.show()

# Statistiche
peak_gdp_pc = fiscal_df.loc[fiscal_df['PIL pro capite (USD)'].idxmax()]
current_gdp_pc = fiscal_df.iloc[-1]
print(f"\n⬆️ Picco PIL pro capite: ${peak_gdp_pc['PIL pro capite (USD)']:.0f} nel {peak_gdp_pc['Anno']:.0f}")
print(f"📊 PIL pro capite attuale ({current_gdp_pc['Anno']:.0f}): ${current_gdp_pc['PIL pro capite (USD)']:.0f}")
print(f"📉 Riduzione: {((current_gdp_pc['PIL pro capite (USD)'] / peak_gdp_pc['PIL pro capite (USD)']) - 1) * 100:.1f}%")

## 4. Struttura Settoriale dell'Economia
Analisi della composizione settoriale del PIL e dell'impatto della crisi sui diversi settori.

In [ ]:
# Estrazione dati settoriali (% del PIL)
agriculture_data = df[df['Series Code'] == 'NV.AGR.TOTL.ZS'].iloc[0]
industry_data = df[df['Series Code'] == 'NV.IND.TOTL.ZS'].iloc[0]
services_data = df[df['Series Code'] == 'NV.SRV.TOTL.ZS'].iloc[0]

# Preparazione dati per anni recenti
recent_years = ['2000', '2005', '2010', '2015', '2019', '2020']
sector_data = []

for year in recent_years:
    col = f"{year} [YR{year}]"
    if col in df.columns:
        agr = agriculture_data[col]
        ind = industry_data[col]
        srv = services_data[col]
        
        if all(v != '..' for v in [agr, ind, srv]):
            sector_data.append({
                'Anno': year,
                'Agricoltura': float(agr),
                'Industria': float(ind),
                'Servizi': float(srv)
            })

sector_df = pd.DataFrame(sector_data)

# Grafico a barre impilate interattivo
fig_sectors = go.Figure()

fig_sectors.add_trace(go.Bar(
    x=sector_df['Anno'],
    y=sector_df['Agricoltura'],
    name='Agricoltura',
    marker_color='#27ae60',
    hovertemplate='<b>%{x}</b><br>Agricoltura: %{y:.1f}%<extra></extra>'
))

fig_sectors.add_trace(go.Bar(
    x=sector_df['Anno'],
    y=sector_df['Industria'],
    name='Industria',
    marker_color='#e67e22',
    hovertemplate='<b>%{x}</b><br>Industria: %{y:.1f}%<extra></extra>'
))

fig_sectors.add_trace(go.Bar(
    x=sector_df['Anno'],
    y=sector_df['Servizi'],
    name='Servizi',
    marker_color='#3498db',
    hovertemplate='<b>%{x}</b><br>Servizi: %{y:.1f}%<extra></extra>'
))

fig_sectors.update_layout(
    title={
        'text': 'Composizione Settoriale del PIL Siriano (% del PIL)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    barmode='stack',
    xaxis_title='Anno',
    yaxis_title='Percentuale del PIL (%)',
    height=600,
    template='plotly_white',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)'),
    hovermode='x unified'
)

fig_sectors.show()

print("\n🏭 Composizione settoriale più recente:")
if len(sector_df) > 0:
    latest = sector_df.iloc[-1]
    print(f"Anno {latest['Anno']}: Agricoltura {latest['Agricoltura']:.1f}%, Industria {latest['Industria']:.1f}%, Servizi {latest['Servizi']:.1f}%")

## 5. Dashboard Indicatori Socioeconomici
Una panoramica completa degli indicatori chiave per valutare la situazione macroeconomica complessiva.

In [ ]:
# Dashboard riassuntivo con indicatori chiave
indicators = {
    'Accesso elettricità (%)': 'EG.ELC.ACCS.ZS',
    'Mortalità infantile (‰)': 'SP.DYN.IMRT.IN',
    'Aspettativa di vita (anni)': 'SP.DYN.LE00.IN',
    'Alfabetizzazione (%)': 'SE.ADT.LITR.ZS',
    'Disoccupazione (%)': 'SL.UEM.TOTL.ZS'
}

# Creazione subplot con indicatori multipli
fig_dashboard = make_subplots(
    rows=3, cols=2,
    subplot_titles=list(indicators.keys()),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

colors = ['#e74c3c', '#3498db', '#27ae60', '#f39c12', '#9b59b6']
row_col_positions = [(1,1), (1,2), (2,1), (2,2), (3,1)]

for idx, (indicator_name, indicator_code) in enumerate(indicators.items()):
    indicator_data = df[df['Series Code'] == indicator_code]
    
    if len(indicator_data) > 0:
        indicator_data = indicator_data.iloc[0]
        values = []
        years_list = []
        
        for col in df.columns:
            if '[YR' in col:
                year = col.split('[')[0].strip()
                val = indicator_data[col]
                if val != '..':
                    years_list.append(int(year))
                    values.append(float(val))
        
        if values:
            row, col = row_col_positions[idx]
            fig_dashboard.add_trace(
                go.Scatter(
                    x=years_list,
                    y=values,
                    mode='lines+markers',
                    name=indicator_name,
                    line=dict(color=colors[idx], width=2),
                    marker=dict(size=4),
                    showlegend=False,
                    hovertemplate='%{x}: %{y:.1f}<extra></extra>'
                ),
                row=row, col=col
            )

fig_dashboard.update_layout(
    title={
        'text': 'Dashboard Indicatori Socioeconomici della Siria',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 22, 'color': '#2c3e50', 'family': 'Arial Black'}
    },
    height=1000,
    template='plotly_white',
    hovermode='x'
)

# Aggiunta annotazione per il conflitto
fig_dashboard.add_vline(
    x=2011, 
    line_dash="dash", 
    line_color="red", 
    opacity=0.5,
    annotation_text="Inizio conflitto",
    annotation_position="top"
)

fig_dashboard.show()

print("✅ Dashboard degli indicatori socioeconomici creata con successo!")

## 6. Heatmap Evoluzione Economica
Visualizzazione comparativa dell'evoluzione di più indicatori economici nel tempo.

In [ ]:
# Mappa di calore dell'evoluzione economica
key_metrics = {
    'PIL (var %)': 'NY.GDP.MKTP.KD.ZG',
    'Esportazioni (% PIL)': 'NE.EXP.GNFS.ZS',
    'Importazioni (% PIL)': 'NE.IMP.GNFS.ZS',
    'Investimenti (% PIL)': 'NE.GDI.TOTL.ZS',
    'Spesa pubblica (% PIL)': 'NE.CON.GOVT.ZS',
    'Agricoltura (% PIL)': 'NV.AGR.TOTL.ZS'
}

# Raccolta dati per heatmap (2000-2020)
heatmap_years = range(2000, 2021)
heatmap_data = []
metric_names = []

for metric_name, metric_code in key_metrics.items():
    metric_row = df[df['Series Code'] == metric_code]
    if len(metric_row) > 0:
        metric_row = metric_row.iloc[0]
        row_values = []
        
        for year in heatmap_years:
            col = f"{year} [YR{year}]"
            if col in df.columns:
                val = metric_row[col]
                if val != '..':
                    row_values.append(float(val))
                else:
                    row_values.append(None)
            else:
                row_values.append(None)
        
        if any(v is not None for v in row_values):
            heatmap_data.append(row_values)
            metric_names.append(metric_name)

# Normalizzazione dei dati per la visualizzazione
heatmap_normalized = []
for row in heatmap_data:
    row_clean = [v for v in row if v is not None]
    if row_clean:
        min_val = min(row_clean)
        max_val = max(row_clean)
        if max_val != min_val:
            normalized = [(v - min_val) / (max_val - min_val) * 100 if v is not None else None for v in row]
        else:
            normalized = [50 if v is not None else None for v in row]
        heatmap_normalized.append(normalized)

# Creazione heatmap
fig_heatmap = go.Figure(data=go.Heatmap(
    z=heatmap_normalized,
    x=list(heatmap_years),
    y=metric_names,
    colorscale='RdYlGn',
    hovertemplate='<b>%{y}</b><br>Anno: %{x}<br>Valore norm.: %{z:.1f}<extra></extra>',
    colorbar=dict(title="Scala<br>normalizzata")
))

fig_heatmap.update_layout(
    title={
        'text': 'Evoluzione Indicatori Economici Chiave (2000-2020)<br><sub>Valori normalizzati 0-100 per confronto</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    xaxis_title='Anno',
    yaxis_title='Indicatore',
    height=600,
    template='plotly_white'
)

fig_heatmap.show()

print("🔥 Heatmap degli indicatori economici creata!")

## Conclusioni

L'analisi dei dati macroeconomici della Siria rivela:

1. **PIL**: Crollo drammatico dal 2011, con il PIL che è passato da oltre $60 miliardi a meno di $20 miliardi
2. **Inflazione**: Aumento esponenziale dell'indice dei prezzi, con un CPI che ha raggiunto oltre 600 (base 2010=100)
3. **PIL pro capite**: Riduzione drastica del benessere individuale, con valori tornati ai livelli degli anni '70
4. **Struttura economica**: Crescente peso dell'agricoltura nell'economia a discapito dell'industria e dei servizi
5. **Indicatori sociali**: Deterioramento significativo degli indicatori di sviluppo umano

Questi dati sottolineano l'enorme impatto del conflitto sull'economia siriana e la necessità di un percorso di ricostruzione strutturale.